<a href="https://colab.research.google.com/github/kanicaanand/HCV-MARL-Feature-Selection/blob/main/HCV-MARL-FeatureSelection-March%202026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque, Counter
import time
import pickle
import warnings
from copy import deepcopy

warnings.filterwarnings('ignore')
device = torch.device('cpu') # Explicitly set to CPU
print(f"🖥️ Using device: {device}")
print("Explicitly using CPU")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Create directories
os.makedirs('saved_models', exist_ok=True)
os.makedirs('plots_multi_agent_final', exist_ok=True)

print("✅ CPU-optimized environment ready!")

file_path= "dataset bakshi nagar 1.xlsx"
def preprocess_data():
    # Preprocess HCV dataset
    print(f"📁 Loading data from: {file_path}")
    df = pd.read_excel(file_path)
    print("Available columns in dataset:", df.columns.tolist())

    # Map target variable
    df['Status'] = df['Status'].map({
        'HCV RNA Detected': 1,
        'Negative': 0,
        'Target Not Detected': 0,
        '<Titer Min': 0
    })

    # Define expected features (clinical biomarkers only - no Age)
    expected_features = ['Total Protein', 'Albumin', 'Globulin',
                         'ALP', 'SGOT', 'SGPT', 'GGT', 'Bilrubin']
    features = [f for f in expected_features if f in df.columns]

    if len(features) < len(expected_features):
        print("⚠️ Warning: Some features not found. Using:", features)
    if not features:
        raise ValueError("❌ No valid features found in dataset.")

    print("NaN counts before imputation:\n", df[features].isna().sum())

    # Impute missing values with median
    for feature in features:
        df[feature] = df[feature].fillna(df[feature].median())

    if df[features].isna().any().any():
        raise ValueError("❌ NaN values still present after imputation.")

    # Drop rows with NaN in 'Status'
    df = df.dropna(subset=['Status'])

    # Prepare features and target
    X = df[features].values
    y = df['Status'].values

    # Feature scaling
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)

    print(f"✅ Data preprocessing completed!")
    print(f"📊 Final dataset: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"🎯 Class distribution: {np.unique(y, return_counts=True)}")

    return X, y, features, df

def average_results_across_runs(all_runs_results):
    """Average results across multiple runs"""
    if not all_runs_results:
        return None

    agent_names = list(all_runs_results[0].keys())
    averaged_results = {}

    for agent in agent_names:
        averaged_results[agent] = {}

        # Get classifier names
        classifier_names = []
        for key in all_runs_results[0][agent].keys():
            if key not in ['features', 'n_features'] and isinstance(all_runs_results[0][agent][key], dict):
                classifier_names.append(key)

        # Average performance metrics
        for clf in classifier_names:
            metrics = ['accuracy', 'precision', 'recall', 'f1', 'f1_cv']
            averaged_results[agent][clf] = {}

            for metric in metrics:
                values = []
                for run_result in all_runs_results:
                    if clf in run_result[agent] and metric in run_result[agent][clf]:
                        values.append(run_result[agent][clf][metric])

                if values:
                    averaged_results[agent][clf][metric] = {
                        'mean': np.mean(values),
                        'std': np.std(values),
                        'min': np.min(values),
                        'max': np.max(values)
                    }

        # Most frequent feature selection
        feature_selections = []
        for run_result in all_runs_results:
            if 'features' in run_result[agent] and run_result[agent]['features']:
                feature_selections.append(tuple(sorted(run_result[agent]['features'])))

        # Find most common feature combination
        if feature_selections:
            most_common = Counter(feature_selections).most_common(1)[0][0]
            averaged_results[agent]['features'] = list(most_common)
            averaged_results[agent]['n_features'] = len(most_common)
        else:
            averaged_results[agent]['features'] = []
            averaged_results[agent]['n_features'] = 0

    return averaged_results

print("✅ Preprocessing and averaging functions defined!")


class FeatureSelectionEnv:
    """RL Environment for Feature Selection"""

    def __init__(self, X, y, penalty_config=None):
        self.X = X
        self.y = y
        self.n_features = X.shape[1]
        self.state = np.zeros(self.n_features, dtype=int)
        self.model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced')
        self.penalty_config = penalty_config # Store penalty configuration

    def reset(self):
        self.state = np.zeros(self.n_features, dtype=int)
        return self.state.copy()

    def step(self, action, agent_name):
        feature_idx = action // 2
        include = action % 2
        self.state[feature_idx] = include
        selected_features = np.where(self.state == 1)[0]

        if len(selected_features) == 0:
            return self.state.copy(), -1, False, 0

        X_subset = self.X[:, selected_features]
        self.model.fit(X_subset, self.y)
        y_pred = self.model.predict(X_subset)
        f1 = f1_score(self.y, y_pred, zero_division=0)

        # Get penalty from config or use original defaults
        if self.penalty_config and agent_name in self.penalty_config:
            penalty = self.penalty_config[agent_name]
        else:
            # Fallback to original hardcoded logic if no specific config provided
            penalty = 0.04 if agent_name == 'Parsimonious' else 0.02

        reward = f1 - penalty * len(selected_features)

        if len(selected_features) > 4:
            reward -= 0.1 * (len(selected_features) - 4)

        done = False
        return self.state.copy(), reward, done, f1

class QLearningAgent:
    def __init__(self, epsilon, alpha, gamma, n_actions):
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.n_actions = n_actions
        self.q_table = {}
        self.epsilon_decay = 0.995
        self.min_epsilon = 0.1
        self.max_q_table_size = 5000

    def get_action(self, state):
        state_tuple = tuple(state)
        if state_tuple not in self.q_table:
            self.q_table[state_tuple] = np.zeros(self.n_actions)

        if len(self.q_table) > self.max_q_table_size:
            q_sums = {s: np.sum(np.abs(qs)) for s, qs in self.q_table.items()}
            sorted_states = sorted(q_sums.items(), key=lambda x: x[1])
            for s, _ in sorted_states[:len(self.q_table) - self.max_q_table_size]:
                del self.q_table[s]

        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.q_table[state_tuple])

    def update(self, state, action, reward, next_state):
        state_tuple = tuple(state)
        next_state_tuple = tuple(next_state)

        if state_tuple not in self.q_table:
            self.q_table[state_tuple] = np.zeros(self.n_actions)
        if next_state_tuple not in self.q_table:
            self.q_table[next_state_tuple] = np.zeros(self.n_actions)

        current_q = self.q_table[state_tuple][action]
        next_max_q = np.max(self.q_table[next_state_tuple])
        self.q_table[state_tuple][action] += self.alpha * (reward + self.gamma * next_max_q - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

class DQN(nn.Module):
    def __init__(self, n_features, n_actions):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions)
        )

    def forward(self, x):
        return self.net(x)

class DQNAgent:
    def __init__(self, n_features, n_actions, epsilon, lr, gamma):
        self.n_features = n_features
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.gamma = gamma
        self.device = device

        self.model = DQN(n_features, n_actions).to(self.device)
        self.target_model = DQN(n_features, n_actions).to(self.device)
        self.target_model.load_state_dict(self.model.state_dict())
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.memory = deque(maxlen=2000)
        self.epsilon_decay = 0.995
        self.min_epsilon = 0.1
        # Fixed batch size for CPU
        self.batch_size = 64 # Was: 128 if torch.cuda.is_available() else 64
        self.target_update_freq = 100
        self.steps = 0
        self.loss_history = []

    def get_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.model(state_tensor)
        return q_values.argmax().item() # Removed extra parenthesis

    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state.copy(), action, reward, next_state.copy(), done))

    def train(self):
        if len(self.memory) < self.batch_size:
            return
        batch_indices = np.random.choice(len(self.memory), self.batch_size, replace=False)
        batch = [self.memory[i] for i in batch_indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        current_q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_values = self.target_model(next_states).max(1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values

        loss = nn.SmoothL1Loss()(current_q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()

        self.steps += 1
        self.loss_history.append(loss.item())

        if self.steps % self.target_update_freq == 0:
            self.target_model.load_state_dict(self.model.state_dict())


    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

class RandomAgent:
    def __init__(self, n_actions):
        self.n_actions = n_actions

    def get_action(self, state):
        return np.random.randint(self.n_actions)

    def update(self, state, action, reward, next_state):
        pass

    def store_transition(self, state, action, reward, next_state, done):
        pass

    def train(self):
        pass

    def decay_epsilon(self):
        pass

print("🧠 All agent classes defined!")


def train_agents(X, y, features, n_episodes=200, max_steps=15, run_seed=42, penalty_config=None):
  # CPU-accelerated agent training
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    env = FeatureSelectionEnv(X, y, penalty_config=penalty_config)
    n_actions = 2 * len(features)

    agents = {
        'Greedy': QLearningAgent(epsilon=0.2, alpha=0.15, gamma=0.9, n_actions=n_actions),
        'Exploratory': QLearningAgent(epsilon=0.6, alpha=0.05, gamma=0.9, n_actions=n_actions),
        'Parsimonious': DQNAgent(n_features=len(features), n_actions=n_actions, epsilon=0.4, lr=0.001, gamma=0.9),
        'Random': RandomAgent(n_actions=n_actions)
    }

    rewards = {name: [] for name in agents}
    accuracies = {name: [] for name in agents}
    feature_selections = {name: [] for name in agents}
    feature_evolution = {name: [] for name in agents}
    agent_training_times = {name: 0.0 for name in agents} # NEW: Initialize agent training times

    for ep in range(n_episodes):
        for name, agent in agents.items():
            state = env.reset()
            ep_reward = 0
            ep_f1 = 0

            start_agent_train_episode_time = time.time() # NEW: Start timing for agent's episode

            for step in range(max_steps):
                action = agent.get_action(state)
                next_state, reward, done, f1 = env.step(action, name)

                if name != 'Random':
                    if name == 'Parsimonious':
                        agent.store_transition(state, action, reward, next_state, done)
                        agent.train()
                    else:
                        agent.update(state, action, reward, next_state)

                state = next_state
                ep_reward += reward
                ep_f1 = f1

            end_agent_train_episode_time = time.time() # NEW: End timing for agent's episode
            agent_training_times[name] += (end_agent_train_episode_time - start_agent_train_episode_time) # NEW: Accumulate time

            rewards[name].append(ep_reward)
            accuracies[name].append(ep_f1)
            feature_selections[name].append(state.copy())
            agent.decay_epsilon()

        if (ep + 1) % 40 == 0:
            print(f"Episode {ep + 1}/{n_episodes} completed.")

    return agents, rewards, accuracies, feature_selections, feature_evolution, agent_training_times # NEW: Return training times

def evaluate_agents_multi_classifier(agents, X_train, y_train, X_test, y_test, feature_selections, features, verbose=True):
    # CPU-optimized multi-classifier evaluation
    classifiers = {
        'Logistic Regression': LogisticRegression(
            solver='liblinear', random_state=42, class_weight='balanced',
            max_iter=2000, n_jobs=-1
        ),
        'Decision Tree': DecisionTreeClassifier(
            random_state=42, class_weight='balanced', max_depth=12
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=100, random_state=42, class_weight='balanced',
            max_depth=12, n_jobs=-1
        ),
        'XGBoost': xgb.XGBClassifier(
            random_state=42, eval_metric='logloss', max_depth=6,
            n_estimators=100, learning_rate=0.1,
            # Fixed to CPU-only tree method
            tree_method='hist',

            n_jobs=-1 # Always use all available CPU cores
        )
    }

    results = {}
    agent_clf_evaluation_times = {name: {} for name in agents} # NEW: Initialize evaluation times

    for agent_name in agents:
        if verbose:
            print(f"\n{'='*60}\n🤖 Evaluating Agent: {agent_name}\n{'='*60}")

        # In evaluate_agents_multi_classifier, feature_selections is the list of final states per episode.
        # We need to take the *last* one to get the final feature selection for this agent/run.
        # OR, if we want to be more robust, we could take the features from the episode
        # that yielded the highest reward/accuracy during training.
        # For now, let's use the features from the very last episode for evaluation.
        if agent_name in feature_selections and len(feature_selections[agent_name]) > 0:
            final_state = feature_selections[agent_name][-1]
        else:
            final_state = np.zeros(len(features), dtype=int)

        selected_features = np.where(final_state == 1)[0]

        if len(selected_features) == 0:
            if verbose:
                print(f"⚠️ No features selected for {agent_name}")
            results[agent_name] = {clf_name: {'accuracy':0,'precision':0,'recall':0,'f1':0,'f1_cv':0} for clf_name in classifiers}
            results[agent_name]['features'] = []
            results[agent_name]['n_features'] = 0
            for clf_name in classifiers:
                agent_clf_evaluation_times[agent_name][clf_name] = 0.0 # NEW: Store 0 for no features
            continue

        feature_names = [features[i] for i in selected_features]
        if verbose:
            print(f"📊 Features ({len(selected_features)}): {feature_names}")

        X_train_sub = X_train[:, selected_features]
        X_test_sub = X_test[:, selected_features]
        results[agent_name] = {'features': feature_names, 'n_features': len(selected_features)}

        if verbose:
            print(f"{'Classifier':<20}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1':<10}{'CV F1':<10}")
            print("-" * 80)

        for clf_name, clf in classifiers.items():
            start_clf_eval_time = time.time() # NEW: Start timing for classifier evaluation
            try:
                f1_cv_scores = cross_val_score(clf, X_train_sub, y_train, cv=5, scoring='f1', n_jobs=-1)
                f1_cv = f1_cv_scores.mean()

                clf.fit(X_train_sub, y_train)
                y_pred = clf.predict(X_test_sub)

                acc = accuracy_score(y_test, y_pred)
                prec = precision_score(y_test, y_pred, zero_division=0)
                rec = recall_score(y_test, y_pred, zero_division=0)
                f1 = f1_score(y_test, y_pred, zero_division=0)

                results[agent_name][clf_name] = {'accuracy':acc, 'precision':prec, 'recall':rec, 'f1':f1, 'f1_cv':f1_cv}

                if verbose:
                    print(f"{clf_name:<20}{acc:<10.4f}{prec:<10.4f}{rec:<10.4f}{f1:<10.4f}{f1_cv:<10.4f}")

            except Exception as e:
                if verbose:
                    print(f"❌ Error with {clf_name}: {e}")
                results[agent_name][clf_name] = {'accuracy':0,'precision':0,'recall':0,'f1':0,'f1_cv':0}
            end_clf_eval_time = time.time() # NEW: End timing for classifier evaluation
            agent_clf_evaluation_times[agent_name][clf_name] = (end_clf_eval_time - start_clf_eval_time) # NEW: Store evaluation time

    return results, agent_clf_evaluation_times # NEW: Return evaluation times

print("🏃‍♂️ Training and evaluation functions defined!")

# --- Global Data Loading and Initial Split for general use --- # NEWLY ADDED BLOCK
print("\n📊 Initial Data Loading and Splitting for general use...")
X, y, features, df_orig = preprocess_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42 # Using a fixed seed for global split
)
print("✅ Initial data split complete.")
# --- End Global Data Loading and Initial Split --- # NEWLY ADDED BLOCK

if __name__ == "__main__":
    try:
        # Configuration
        N_RUNS = 6
        N_EPISODES = 100 # Fixed for CPU

        start_time = time.time()
        print("🚀 CPU-ONLY MULTI-AGENT RL FOR HCV DETECTION") # Updated print
        print("=" * 80)
        print(f"🖥️ Running on: {device}")

        # Load data - Removed this block as it's now handled globally above
        # print("\n📊 Step 1: Loading and preprocessing data...")
        # X, y, features, df_orig = preprocess_data()

        # Check for cached results
        # Updated results file name to reflect CPU-only
        results_file = f'hcv_rl_multirun_{N_RUNS}runs_cpu.pkl'
        cached_file_path = os.path.join('saved_models', results_file)

        if os.path.exists(cached_file_path):
            print(f"⚡ Loading cached CPU results...") # Updated print
            with open(cached_file_path, 'rb') as f:
                cached_data = pickle.load(f)
            averaged_results = cached_data['averaged_results']
            all_runs_results = cached_data['all_runs_results']
            # NEW: Load training histories if available
            all_runs_training_histories = cached_data.get('all_runs_training_histories', [])
        else:
            print(f"\n🏃‍♂️ Step 2: Running {N_RUNS} CPU-only experiments...") # Updated print

            all_runs_results = []
            all_runs_training_histories = [] # NEW: Initialize list for training histories

            for run_idx in range(N_RUNS):
                print(f"\n🚀 CPU Run {run_idx + 1}/{N_RUNS}") # Updated print
                print("-" * 50)

                run_seed = 42 + run_idx * 1000

                # Split data - This re-splits for each run in the loop, which is fine.
                # The global X_train, y_train etc. are for cells outside this loop.
                X_train_run, X_test_run, y_train_run, y_test_run = train_test_split(
                    X, y, test_size=0.2, stratify=y, random_state=run_seed
                )

                # Train agents (no penalty_config passed here, so it uses defaults)
                print("🧠 Training agents on CPU...") # Updated print
                agents, rewards, accuracies, feature_selections, feature_evolution, agent_training_times = train_agents(
                    X_train_run, y_train_run, features, n_episodes=N_EPISODES, max_steps=15, run_seed=run_seed
                ) # NEW: Capture agent_training_times

                # Evaluate
                print("🧪 Evaluating with CPU-optimized classifiers...") # Updated print
                results, agent_clf_evaluation_times = evaluate_agents_multi_classifier(
                    agents, X_train_run, y_train_run, X_test_run, y_test_run, feature_selections, features, verbose=False
                ) # NEW: Capture agent_clf_evaluation_times

                all_runs_results.append(results)
                # NEW: Store training histories and timing data for this run
                all_runs_training_histories.append({
                    'rewards_per_episode': rewards,
                    'accuracies_per_episode': accuracies,
                    'feature_selections_per_episode': feature_selections,
                    'agent_training_times': agent_training_times, # NEW: Add agent training times
                    'agent_clf_evaluation_times': agent_clf_evaluation_times # NEW: Add classifier evaluation times
                })

                # Print run summary
                print(f"📊 CPU Run {run_idx + 1} Summary:") # Updated print
                for agent in results:
                    if results[agent]['n_features'] > 0:
                        best_f1 = 0
                        best_clf = ""
                        for clf in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
                            if clf in results[agent] and results[agent][clf]['f1'] > best_f1:
                                best_f1 = results[agent][clf]['f1']
                                best_clf = clf
                        print(f"   {agent}: {best_f1:.4f} F1 ({best_clf}), {results[agent]['n_features']} features")


            # Average results
            print(f"\n📊 Computing averaged results across {N_RUNS} CPU runs...") # Updated print
            averaged_results = average_results_across_runs(all_runs_results)

            # Save results
            results_data = {
                'averaged_results': averaged_results,
                'all_runs_results': all_runs_results,
                'all_runs_training_histories': all_runs_training_histories, # NEW: Save training histories
                'n_runs': N_RUNS,
                'n_episodes': N_EPISODES,
                'features': features,
                'device': str(device),
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
            }

            with open(cached_file_path, 'wb') as f:
                pickle.dump(results_data, f)
            print(f"💾 CPU results saved to {cached_file_path}") # Updated print

        # Display results
        print(f"\n📊 CPU-ONLY RESULTS ACROSS {N_RUNS} RUNS:") # Updated print
        print("=" * 80)

        best_f1_mean = 0
        best_combination = ""

        for agent_name in averaged_results:
            if averaged_results[agent_name]['n_features'] > 0:
                print(f"\n🤖 {agent_name.upper()}:")
                print(f"   Selected Features ({averaged_results[agent_name]['n_features']}): {averaged_results[agent_name]['features']}")
                print(f"   {'Classifier':<20} {'F1 Mean\u00b1Std':<15} {'Accuracy':<12} {'CV F1':<10}")
                print("-" * 65)

                for clf_name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
                    if clf_name in averaged_results[agent_name]:
                        f1_stats = averaged_results[agent_name][clf_name]['f1']
                        acc_stats = averaged_results[agent_name][clf_name]['accuracy']
                        cv_stats = averaged_results[agent_name][clf_name]['f1_cv']


                        gpu_marker = "  " # Always print empty marker for CPU-only
                        print(f"{gpu_marker} {clf_name:<18} {f1_stats['mean']:.3f}\u00b1{f1_stats['std']:.3f}     "
                              f"{acc_stats['mean']:.3f}      {cv_stats['mean']:.3f}")

                        # Track best combination
                        if f1_stats['mean'] > best_f1_mean:
                            best_f1_mean = f1_stats['mean']
                            best_combination = f"{agent_name} + {clf_name}"

        print(f"\n🏆 OVERALL BEST CPU-ONLY COMBINATION:") # Updated print
        print(f"🥇 Best F1 Score: {best_f1_mean:.4f} ({best_combination})")

        # Performance summary
        total_time = time.time() - start_time
        print(f"\n⏱️ Performance Summary (CPU-ONLY):") # Updated print
        print(f"   Total runtime: {total_time:.2f} seconds")
        print(f"   Average time per run: {total_time/N_RUNS:.2f} seconds")
        print(f"   Device used: {device}")


        print(f"\n🔬 Completed: {N_RUNS} runs \u00d7 4 agents \u00d7 4 classifiers = {N_RUNS * 16} CPU-only evaluations") # Updated print
        print(f"💾 Results cached for visualization pipeline!")

        print(f"\n✅ CPU-only multi-run analysis completed successfully! 🎉") # Updated print

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()



print("🎯 Fixed CPU-optimized complete pipeline ready!")

🖥️ Using device: cpu
Explicitly using CPU
✅ CPU-optimized environment ready!
✅ Preprocessing and averaging functions defined!
🧠 All agent classes defined!
🏃‍♂️ Training and evaluation functions defined!

📊 Initial Data Loading and Splitting for general use...
📁 Loading data from: dataset bakshi nagar 1.xlsx
Available columns in dataset: ['S.no', 'Gender', 'Age', 'Unnamed: 3', 'Status', 'Viral Load', 'Total Protein', 'Albumin', 'Globulin', 'ALP', 'SGOT', 'SGPT', 'GGT', 'Bilrubin']
NaN counts before imputation:
 Total Protein    7
Albumin          7
Globulin         7
ALP              7
SGOT             7
SGPT             7
GGT              7
Bilrubin         8
dtype: int64
✅ Data preprocessing completed!
📊 Final dataset: 938 samples, 8 features
🎯 Class distribution: (array([0., 1.]), array([500, 438]))
✅ Initial data split complete.
🚀 CPU-ONLY MULTI-AGENT RL FOR HCV DETECTION
🖥️ Running on: cpu

🏃‍♂️ Step 2: Running 6 CPU-only experiments...

🚀 CPU Run 1/6
------------------------------

In [ ]:
# F1-score across 6 independent runs
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# f1_scores_df should be available from the previous box plot generation (cell 03d5dbde)
# If not, you may need to re-run cell 03d5dbde first.

plt.figure(figsize=(18, 10))
sns.lineplot(
    data=f1_scores_df,
    x='Run',
    y='F1-Score',
    hue='Agent',
    style='Classifier',
    marker='o', # Add markers to denote individual runs
    palette='tab10'
)
plt.title('F1-Score Trends Across 6 Independent Runs by Agent and Classifier')
plt.xlabel('Experiment Run')
plt.ylabel('F1-Score')
plt.ylim(0.0, 1.05) # Ensure full range is visible
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45) # Rotate x-axis labels if needed
plt.legend(title='Agent & Classifier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("✅ F1-score line chart generated!")

In [ ]:
# Feature Selection per run
import pandas as pd
import numpy as np

features_per_run_data = []

for run_idx, run_history in enumerate(all_runs_training_histories):
    run_name = f"Run {run_idx + 1}"
    row_data = {'Run': run_name}

    for agent_name, agent_feature_selections_list in run_history['feature_selections_per_episode'].items():
        # Take the final feature selection state from the last episode of this run for this agent
        final_episode_state = agent_feature_selections_list[-1]
        selected_indices = np.where(final_episode_state == 1)[0]
        selected_feature_names = [features[i] for i in selected_indices]

        row_data[agent_name] = ', '.join(selected_feature_names) if selected_feature_names else 'No Features Selected'

    features_per_run_data.append(row_data)

features_per_run_df = pd.DataFrame(features_per_run_data)
display(features_per_run_df)

print("✅ Feature selection per run displayed!")

In [ ]:
import pandas as pd

# Assuming agent_training_df is available from previous execution (cell 16771648)
# If not, please ensure cell 16771648 has been executed.

# Calculate mean and standard deviation of training times per agent
agent_runtime_summary = agent_training_df.groupby('Agent')['Training Time (s)'].agg(['mean', 'std']).reset_index()
agent_runtime_summary.rename(columns={'mean': 'Mean Training Time (s)', 'std': 'Std Dev Training Time (s)'}, inplace=True)

display(agent_runtime_summary)

print("✅ Agent training runtime summary table displayed!")

In [ ]:
import pandas as pd

# Ensure agent_training_df is available from previous execution (cell 16771648)
# If not, please re-run cell 16771648 first.

print("**Table 1: Agent Training Runtime Summary (Averaged Across Runs)**")
agent_runtime_summary = agent_training_df.groupby('Agent')['Training Time (s)'].agg(['mean', 'std']).reset_index()
agent_runtime_summary.rename(columns={'mean': 'Mean Training Time (s)', 'std': 'Std Dev Training Time (s)'}, inplace=True)
display(agent_runtime_summary)

print("\n**Table 2: Classifier Evaluation Runtime Summary per Agent (Averaged Across Runs)**")
# Ensure clf_evaluation_df is available from previous execution (cell 16771648)
# If not, please re-run cell 16771648 first.
clf_runtime_summary = clf_evaluation_df.groupby(['Agent', 'Classifier'])['Evaluation Time (s)'].agg(['mean', 'std']).reset_index()
clf_runtime_summary.rename(columns={'mean': 'Mean Evaluation Time (s)', 'std': 'Std Dev Evaluation Time (s)'}, inplace=True)
display(clf_runtime_summary)

print("✅ Computational feasibility summary tables generated!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- 1. Process Agent Training Times ---
agent_training_data = []
for run_history in all_runs_training_histories:
    for agent_name, time_taken in run_history['agent_training_times'].items():
        agent_training_data.append({'Agent': agent_name, 'Training Time (s)': time_taken})

agent_training_df = pd.DataFrame(agent_training_data)
# mean_agent_training_times = agent_training_df.groupby('Agent')['Training Time (s)'].agg(['mean', 'std']).reset_index() # Not strictly needed if using errorbar='sd' directly

# --- 2. Process Classifier Evaluation Times ---
clf_evaluation_data = []
for run_history in all_runs_training_histories:
    for agent_name, clf_times in run_history['agent_clf_evaluation_times'].items():
        for clf_name, time_taken in clf_times.items():
            clf_evaluation_data.append({'Agent': agent_name, 'Classifier': clf_name, 'Evaluation Time (s)': time_taken})

clf_evaluation_df = pd.DataFrame(clf_evaluation_data)
# mean_clf_evaluation_times = clf_evaluation_df.groupby(['Agent', 'Classifier'])['Evaluation Time (s)'].agg(['mean', 'std']).reset_index() # Not strictly needed if using errorbar='sd' directly

# --- 3. Plotting Training Times ---
plt.figure(figsize=(10, 6))
# Use the raw dataframe and let seaborn calculate the mean and standard deviation
sns.barplot(data=agent_training_df, x='Agent', y='Training Time (s)', errorbar='sd', capsize=0.1)
plt.title('Average Agent Training Time Across Runs (CPU)')
plt.xlabel('RL Agent')
plt.ylabel('Average Training Time (s)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- 4. Plotting Classifier Evaluation Times ---
plt.figure(figsize=(12, 7))
# Use the raw dataframe and let seaborn calculate the mean and standard deviation
sns.barplot(data=clf_evaluation_df, x='Agent', y='Evaluation Time (s)', hue='Classifier', errorbar='sd', capsize=0.1, palette='viridis')
plt.title('Average Classifier Evaluation Time per Agent Across Runs (CPU)')
plt.xlabel('RL Agent')
plt.ylabel('Average Evaluation Time (s)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Classifier', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\n✅ Computational feasibility plots generated!")

In [ ]:
import pandas as pd

# Ensure agent_training_df is available from previous execution (cell 16771648)
# If not, please re-run cell 16771648 first.

print("**Table 1: Agent Training Runtime Summary (Averaged Across Runs)**")
agent_runtime_summary = agent_training_df.groupby('Agent')['Training Time (s)'].agg(['mean', 'std']).reset_index()
agent_runtime_summary.rename(columns={'mean': 'Mean Training Time (s)', 'std': 'Std Dev Training Time (s)'}, inplace=True)
display(agent_runtime_summary)

print("\n**Table 2: Classifier Evaluation Runtime Summary per Agent (Averaged Across Runs)**")
# Ensure clf_evaluation_df is available from previous execution (cell 16771648)
# If not, please re-run cell 16771648 first.
clf_runtime_summary = clf_evaluation_df.groupby(['Agent', 'Classifier'])['Evaluation Time (s)'].agg(['mean', 'std']).reset_index()
clf_runtime_summary.rename(columns={'mean': 'Mean Evaluation Time (s)', 'std': 'Std Dev Evaluation Time (s)'}, inplace=True)
display(clf_runtime_summary)

print("✅ Computational feasibility summary tables generated!")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data for the accuracy heatmap
accuracy_heatmap_data = []

for agent_name, agent_result in averaged_results.items():
    # Ensure the agent actually selected features and has classifier results
    if agent_result.get('n_features', 0) > 0:
        for clf_name, clf_metrics in agent_result.items():
            if isinstance(clf_metrics, dict) and 'accuracy' in clf_metrics:
                accuracy_heatmap_data.append({
                    'Agent': agent_name,
                    'Classifier': clf_name,
                    'Mean Accuracy': clf_metrics['accuracy']['mean']
                })

accuracy_df = pd.DataFrame(accuracy_heatmap_data)

# Pivot the DataFrame for the heatmap
accuracy_pivot_df = accuracy_df.pivot_table(index='Agent', columns='Classifier', values='Mean Accuracy')

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(accuracy_pivot_df, annot=True, cmap='YlGnBu', fmt=".3f", linewidths=.5)
plt.title('Mean Accuracy for Agent-Classifier Combinations (Averaged Across Runs)')
plt.xlabel('Classifier')
plt.ylabel('RL Agent')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data for the recall heatmap
recall_heatmap_data = []

for agent_name, agent_result in averaged_results.items():
    # Ensure the agent actually selected features and has classifier results
    if agent_result.get('n_features', 0) > 0:
        for clf_name, clf_metrics in agent_result.items():
            if isinstance(clf_metrics, dict) and 'recall' in clf_metrics:
                recall_heatmap_data.append({
                    'Agent': agent_name,
                    'Classifier': clf_name,
                    'Mean Recall': clf_metrics['recall']['mean']
                })

recall_df = pd.DataFrame(recall_heatmap_data)

# Pivot the DataFrame for the heatmap
recall_pivot_df = recall_df.pivot_table(index='Agent', columns='Classifier', values='Mean Recall')

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(recall_pivot_df, annot=True, cmap='YlGnBu', fmt=".3f", linewidths=.5)
plt.title('Mean Recall for Agent-Classifier Combinations (Averaged Across Runs)')
plt.xlabel('Classifier')
plt.ylabel('RL Agent')
plt.tight_layout()
plt.show()

In [ ]:
display(ablation_df)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'averaged_results' and 'features' are available from the previous execution

# Create a list of all unique features
all_features = sorted(list(set(f for agent_data in averaged_results.values() for f in agent_data.get('features', []))))

# Create a DataFrame to store the feature selection matrix
feature_matrix_data = []
agent_names_for_plot = []

for agent_name, agent_result in averaged_results.items():
    if agent_result.get('n_features', 0) > 0:
        row = {feature: 0 for feature in all_features}
        for selected_feature in agent_result['features']:
            row[selected_feature] = 1 # Mark selected features with 1
        feature_matrix_data.append(row)
        agent_names_for_plot.append(agent_name)

feature_selection_df = pd.DataFrame(feature_matrix_data, index=agent_names_for_plot)

# Plot the heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(feature_selection_df, annot=True, cmap='viridis', fmt='g', linewidths=.5)
plt.title('Feature Selection Patterns Across RL Agents (Averaged)')
plt.xlabel('Features')
plt.ylabel('RL Agents')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

print("\n\n📊 Performing Statistical Tests (Paired T-Tests) for F1-Scores:\n")
print("=" * 80)

# Get a list of all agent names and classifier names
agent_names = list(all_runs_results[0].keys())
classifier_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']

# Dictionary to store F1 scores per agent per classifier across all runs
f1_scores_per_run = {clf: {agent: [] for agent in agent_names} for clf in classifier_names}

for run_result in all_runs_results:
    for agent_name in agent_names:
        for clf_name in classifier_names:
            # Ensure the agent and classifier exist and have an f1 score
            if agent_name in run_result and \
               clf_name in run_result[agent_name] and \
               'f1' in run_result[agent_name][clf_name]:
                f1_scores_per_run[clf_name][agent_name].append(run_result[agent_name][clf_name]['f1'])


for clf_name in classifier_names:
    print(f"\n--- Classifier: {clf_name} ---")
    print("Comparison (Agent 1 vs. Agent 2) | T-statistic | P-value | Significance (p < 0.05)")
    print("-" * 90)

    # Compare each agent against every other agent
    for i in range(len(agent_names)):
        for j in range(i + 1, len(agent_names)):
            agent1 = agent_names[i]
            agent2 = agent_names[j]

            scores1 = f1_scores_per_run[clf_name][agent1]
            scores2 = f1_scores_per_run[clf_name][agent2]

            if len(scores1) > 1 and len(scores2) > 1: # Need at least 2 samples for t-test
                t_stat, p_value = stats.ttest_rel(scores1, scores2) # Paired t-test

                significance = "Yes" if p_value < 0.05 else "No"
                print(f"{agent1:<25} vs. {agent2:<25} | {t_stat:^11.3f} | {p_value:^7.3f} | {significance:^22}")
            else:
                print(f"Not enough data for {agent1} vs. {agent2} with {clf_name}")
    print("=" * 90)

print("\n✅ Statistical validation completed!")

In [ ]:
# Sensitivity Analysis for Reward Function Penalties
print("\n\n🧪 Performing Reward Function Sensitivity Analysis\n")
print("=" * 80)

penalty_configurations_for_sensitivity = {
    "Default Penalties": None, # Uses the hardcoded defaults in env (0.02/0.04 for feature penalty)
    "Low Feature Penalty": {'Greedy': 0.01, 'Exploratory': 0.01, 'Parsimonious': 0.02}, # Halve penalties
    "High Feature Penalty": {'Greedy': 0.05, 'Exploratory': 0.05, 'Parsimonious': 0.08} # Increase penalties
}

# Use a smaller number of runs/episodes for sensitivity analysis to make it quicker
N_RUNS_SENSITIVITY = 2
N_EPISODES_SENSITIVITY = 50

all_sensitivity_results = {}

# Ensure X, y, features are available from previous data preprocessing
# If this cell is run independently, you might need to call preprocess_data() again:
# X, y, features, df_orig = preprocess_data()

for config_name, penalty_config in penalty_configurations_for_sensitivity.items():
    print(f"\n--- Running Sensitivity Test: {config_name} ---")
    print("-" * (len(config_name) + 29))

    current_sensitivity_runs_results = []
    current_sensitivity_runs_training_histories = []

    for run_idx in range(N_RUNS_SENSITIVITY):
        print(f"  🚀 Sensitivity Run {run_idx + 1}/{N_RUNS_SENSITIVITY}")
        # Use different seeds for each sensitivity run and inner run
        run_seed = 1000 + run_idx * 100 + sum(map(ord, config_name))

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=run_seed
        )

        agents, rewards, accuracies, feature_selections, feature_evolution = train_agents(
            X_train, y_train, features,
            n_episodes=N_EPISODES_SENSITIVITY, max_steps=15, run_seed=run_seed,
            penalty_config=penalty_config # Pass the current penalty config
        )

        results = evaluate_agents_multi_classifier(
            agents, X_train, y_train, X_test, y_test, feature_selections, features, verbose=False
        )
        current_sensitivity_runs_results.append(results)
        current_sensitivity_runs_training_histories.append({
            'rewards_per_episode': rewards,
            'accuracies_per_episode': accuracies,
            'feature_selections_per_episode': feature_selections
        })

    averaged_sensitivity_results = average_results_across_runs(current_sensitivity_runs_results)
    all_sensitivity_results[config_name] = averaged_sensitivity_results

    # Print summary for this configuration
    print(f"\n📊 Summary for {config_name}:")
    for agent_name in averaged_sensitivity_results:
        if averaged_sensitivity_results[agent_name]['n_features'] > 0:
            print(f"  🤖 {agent_name}:")
            print(f"     Selected Features ({averaged_sensitivity_results[agent_name]['n_features']}): {averaged_sensitivity_results[agent_name]['features']}")
            for clf_name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
                if clf_name in averaged_sensitivity_results[agent_name]:
                    f1_stats = averaged_sensitivity_results[agent_name][clf_name]['f1']
                    print(f"       - {clf_name}: F1 Mean {f1_stats['mean']:.3f} \u00b1 {f1_stats['std']:.3f}")
        else:
            print(f"  🤖 {agent_name}: No features selected.")

print("\n✅ Reward function sensitivity analysis completed. Review the summaries above.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data for plotting the sensitivity analysis
plot_data_sensitivity = []

for config_name, averaged_results_for_config in all_sensitivity_results.items():
    for agent_name, agent_result in averaged_results_for_config.items():
        if agent_result.get('n_features', 0) > 0: # Only include agents that selected features
            for clf_name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
                if clf_name in agent_result:
                    f1_mean = agent_result[clf_name]['f1']['mean']
                    f1_std = agent_result[clf_name]['f1']['std']
                    plot_data_sensitivity.append({
                        'Penalty Configuration': config_name,
                        'Agent': agent_name,
                        'Classifier': clf_name,
                        'F1 Score Mean': f1_mean,
                        'F1 Score Std': f1_std
                    })

performance_sensitivity_df = pd.DataFrame(plot_data_sensitivity)

# Order for plotting configurations
config_order = ["Default Penalties", "Low Feature Penalty", "High Feature Penalty"]
performance_sensitivity_df['Penalty Configuration'] = pd.Categorical(
    performance_sensitivity_df['Penalty Configuration'], categories=config_order, ordered=True
)

# Create a faceted bar plot for F1-score comparison
g = sns.catplot(
    data=performance_sensitivity_df,
    x='Agent',
    y='F1 Score Mean',
    hue='Classifier',
    col='Penalty Configuration',
    kind='bar',
    col_wrap=2, # Wrap columns after 2 plots
    height=5,
    aspect=1.2,
    palette='viridis',
    errorbar='sd', # Show standard deviation as error bars
    sharey=True, # Share y-axis scale across facets
    sharex=False # Allow different x-axis labels if needed, set directly in catplot
)

g.set_axis_labels("Reinforcement Learning Agent", "Average F1-Score (Mean ± Std)")
g.set_titles("Penalty: {col_name}")
g.set_xticklabels(rotation=45, ha='right')
g.tight_layout()
plt.suptitle('Reward Function Sensitivity Analysis: F1-Score Performance', y=1.02) # Adjust suptitle position
plt.show()

In [ ]:
# F1-score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data for the heatmap
heatmap_data = []

for agent_name, agent_result in averaged_results.items():
    # Ensure the agent actually selected features and has classifier results
    if agent_result.get('n_features', 0) > 0:
        for clf_name, clf_metrics in agent_result.items():
            if isinstance(clf_metrics, dict) and 'f1' in clf_metrics:
                heatmap_data.append({
                    'Agent': agent_name,
                    'Classifier': clf_name,
                    'Mean F1-Score': clf_metrics['f1']['mean']
                })

f1_heatmap_df = pd.DataFrame(heatmap_data)

# Pivot the DataFrame for the heatmap
f1_pivot_df = f1_heatmap_df.pivot_table(index='Agent', columns='Classifier', values='Mean F1-Score')

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(f1_pivot_df, annot=True, cmap='YlGnBu', fmt=".3f", linewidths=.5)
plt.title('Mean F1-Score for Agent-Classifier Combinations (Averaged Across Runs)')
plt.xlabel('Classifier')
plt.ylabel('RL Agent')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data for the precision heatmap
precision_heatmap_data = []

for agent_name, agent_result in averaged_results.items():
    # Ensure the agent actually selected features and has classifier results
    if agent_result.get('n_features', 0) > 0:
        for clf_name, clf_metrics in agent_result.items():
            if isinstance(clf_metrics, dict) and 'precision' in clf_metrics:
                precision_heatmap_data.append({
                    'Agent': agent_name,
                    'Classifier': clf_name,
                    'Mean Precision': clf_metrics['precision']['mean']
                })

precision_df = pd.DataFrame(precision_heatmap_data)

# Pivot the DataFrame for the heatmap
precision_pivot_df = precision_df.pivot_table(index='Agent', columns='Classifier', values='Mean Precision')

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(precision_pivot_df, annot=True, cmap='YlGnBu', fmt=".3f", linewidths=.5)
plt.title('Mean Precision for Agent-Classifier Combinations (Averaged Across Runs)')
plt.xlabel('Classifier')
plt.ylabel('RL Agent')
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Initialize a counter for all selected features
all_selected_features_counter = Counter()

# Loop through each run's training history
for run_history in all_runs_training_histories:
    # Loop through each agent's feature selections within that run
    for agent_name, agent_feature_selections_list in run_history['feature_selections_per_episode'].items():
        # Loop through each episode's final feature selection for the agent
        for episode_feature_state in agent_feature_selections_list:
            # Get the indices of the selected features (where state is 1)
            selected_indices = np.where(episode_feature_state == 1)[0]
            # Convert indices to actual feature names
            selected_feature_names = [features[i] for i in selected_indices]
            # Update the counter
            all_selected_features_counter.update(selected_feature_names)

# Convert the Counter to a DataFrame for plotting
feature_importance_df = pd.DataFrame(
    all_selected_features_counter.items(), columns=['Feature', 'Selection Frequency']
).sort_values(by='Selection Frequency', ascending=False)

print("\nGlobal Feature Importance based on Selection Frequency:")
print(feature_importance_df)

# Plotting the feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Selection Frequency', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Global Feature Importance based on Selection Frequency (All Agents, All Runs)')
plt.xlabel('Total Selection Count')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Prepare data structure for plotting
plot_data_performance_curves = []

# Assuming N_EPISODES is the number of episodes per run
n_episodes = N_EPISODES # This comes from the main script execution
agent_names = list(all_runs_training_histories[0]['accuracies_per_episode'].keys())

for agent_name in agent_names:
    # Collect F1-scores for this agent across all runs and all episodes
    agent_f1_scores_per_episode = []
    for run_history in all_runs_training_histories:
        if agent_name in run_history['accuracies_per_episode']:
            agent_f1_scores_per_episode.append(run_history['accuracies_per_episode'][agent_name])

    # Convert to numpy array for easy averaging
    agent_f1_scores_per_episode_np = np.array(agent_f1_scores_per_episode)

    # Calculate mean and standard deviation across runs for each episode
    mean_f1_per_episode = np.mean(agent_f1_scores_per_episode_np, axis=0)
    std_f1_per_episode = np.std(agent_f1_scores_per_episode_np, axis=0)

    # Add to plot data
    for ep in range(n_episodes):
        plot_data_performance_curves.append({
            'Agent': agent_name,
            'Episode': ep + 1,
            'Mean F1-Score': mean_f1_per_episode[ep],
            'Std F1-Score': std_f1_per_episode[ep]
        })

performance_curves_df = pd.DataFrame(plot_data_performance_curves)

# Plotting the performance curves
plt.figure(figsize=(14, 8))
sns.lineplot(
    data=performance_curves_df,
    x='Episode',
    y='Mean F1-Score',
    hue='Agent',
    errorbar='sd', # Show standard deviation as shaded area
    linewidth=2.5,
    palette='tab10'
)

plt.title('Agent Performance (F1-Score) Across Training Episodes (Averaged Over Runs)')
plt.xlabel('Training Episode')
plt.ylabel('Average F1-Score (Mean \u00b1 Std)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Agent')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Prepare data for the ablation table
ablation_data = []

for agent_name, agent_result in averaged_results.items():
    best_f1_mean = 0
    best_f1_std = 0

    # Find the best F1-score and its std across all classifiers for this agent
    for clf_name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
        if clf_name in agent_result:
            current_f1_mean = agent_result[clf_name]['f1']['mean']
            current_f1_std = agent_result[clf_name]['f1']['std']
            if current_f1_mean > best_f1_mean:
                best_f1_mean = current_f1_mean
                best_f1_std = current_f1_std

    ablation_data.append({
        'Agent Policy': agent_name,
        'Best Mean F1-Score': f'{best_f1_mean:.3f} ± {best_f1_std:.3f}',
        'Mean # Features Selected': agent_result.get('n_features', 0),
        'Most Frequent Selected Features': ', '.join(agent_result.get('features', []))
    })

# Create DataFrame and display
ablation_df = pd.DataFrame(ablation_data)
display(ablation_df)

In [ ]:
import pandas as pd

# Ensure agent_training_df and clf_evaluation_df are available from previous execution (cell 16771648)
# If not, please re-run cell 16771648 first.

print("**Table 1: Agent Training Runtime Summary (Averaged Across Runs)**")
agent_runtime_summary = agent_training_df.groupby('Agent')['Training Time (s)'].agg(['mean', 'std']).reset_index()
agent_runtime_summary.rename(columns={'mean': 'Mean Training Time (s)', 'std': 'Std Dev Training Time (s)'}, inplace=True)
display(agent_runtime_summary)

print("\n**Table 2: Classifier Evaluation Runtime Summary per Agent (Averaged Across Runs)**")
clf_runtime_summary = clf_evaluation_df.groupby(['Agent', 'Classifier'])['Evaluation Time (s)'].agg(['mean', 'std']).reset_index()
clf_runtime_summary.rename(columns={'mean': 'Mean Evaluation Time (s)', 'std': 'Std Dev Evaluation Time (s)'}, inplace=True)
display(clf_runtime_summary)

print("✅ Computational feasibility summary tables generated!")